# 🛡️ MindGuard — Train VAE Crisis Detector (Colab)

This notebook trains a **Variational Autoencoder (VAE)** for **crisis detection** using reconstruction error anomaly detection.

### How it works
1. The VAE is trained **only on non-crisis text** (TF-IDF vectorized)
2. At inference, crisis text produces **higher reconstruction error** because the model hasn't learned to reconstruct crisis patterns
3. A threshold (derived from the 95th percentile of validation errors) separates crisis from non-crisis

### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** (optional — CPU also works, ~2 min)
2. Upload `crisis_labeled_text.csv` from your project's `data/processed/` folder
3. Run all cells in order
4. Download the output files at the end

**Estimated time:** ~2–5 minutes (CPU is fine for this model)

## 1. Install Dependencies

In [ ]:
!pip install -q torch scikit-learn pandas numpy mlflow

## 2. Upload Your Crisis Dataset

Upload `crisis_labeled_text.csv` from your project's `data/processed/` directory.

**Expected format:** CSV with columns `text` and `label` (1=crisis, 0=non-crisis)

In [ ]:
import os
from pathlib import Path

# Try Colab upload first, fall back to local path
DATA_PATH = "crisis_labeled_text.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("📤 Upload crisis_labeled_text.csv from data/processed/")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {DATA_PATH}")
    except ImportError:
        raise FileNotFoundError(
            "Place crisis_labeled_text.csv in the current directory "
            "or run in Google Colab for upload support."
        )
else:
    print(f"✅ Found dataset at: {DATA_PATH}")

## 3. Configuration

In [ ]:
import random
import numpy as np
import torch

# ---------- You can tweak these ----------
MAX_FEATURES = 4000        # TF-IDF vocabulary size
HIDDEN_DIM = 256           # VAE encoder/decoder hidden dimension
LATENT_DIM = 32            # VAE latent space dimension
BATCH_SIZE = 64
EPOCHS = 8
LEARNING_RATE = 1e-3
VALIDATION_RATIO = 0.2     # Fraction of non-crisis data for validation
THRESHOLD_PERCENTILE = 95  # Percentile of validation errors for threshold
SEED = 42
# -----------------------------------------

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config: hidden={HIDDEN_DIM}, latent={LATENT_DIM}, epochs={EPOCHS}, lr={LEARNING_RATE}")

## 4. Load & Explore the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nSample rows:")
df.head()

## 5. Split Data: Train on Non-Crisis Only

The key insight of VAE anomaly detection: we train **only on non-crisis data**, so the model learns the "normal" distribution. Crisis text will have **higher reconstruction error** because it doesn't fit the learned pattern.

In [ ]:
# Clean text column
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()

# Ensure label is integer
CRISIS_LABELS = {"suicidal", "suicide", "1", "true"}
if df["label"].dtype == object:
    df["label"] = df["label"].apply(
        lambda v: 1 if str(v).strip().lower() in CRISIS_LABELS else 0
    )
df["is_crisis"] = df["label"].astype(int)

non_crisis = df[df["is_crisis"] == 0].copy()
crisis = df[df["is_crisis"] == 1].copy()

print(f"Non-crisis samples: {len(non_crisis):,}")
print(f"Crisis samples:     {len(crisis):,}")

# Split non-crisis into train/validation
shuffled = non_crisis.sample(frac=1.0, random_state=SEED)
split_idx = int(len(shuffled) * (1.0 - VALIDATION_RATIO))
split_idx = max(1, min(split_idx, len(shuffled) - 1))
train_nc = shuffled.iloc[:split_idx]
val_nc = shuffled.iloc[split_idx:]

print(f"\nNon-crisis train: {len(train_nc):,}")
print(f"Non-crisis val:   {len(val_nc):,}")

## 6. TF-IDF Vectorization

Convert text to numerical features using TF-IDF. The vocabulary is fitted only on training data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
x_train = vectorizer.fit_transform(train_nc["text"]).toarray().astype(np.float32)
x_val = vectorizer.transform(val_nc["text"]).toarray().astype(np.float32)
x_full = vectorizer.transform(df["text"]).toarray().astype(np.float32)

input_dim = x_train.shape[1]
print(f"TF-IDF input dimension: {input_dim}")
print(f"Train shape: {x_train.shape}")
print(f"Val shape:   {x_val.shape}")
print(f"Full shape:  {x_full.shape}")

## 7. Define VAE Architecture

The TextVAE uses:
- **Encoder**: Linear → ReLU → produces mean (μ) and log-variance (log σ²)
- **Reparameterization trick**: z = μ + ε × σ (enables backpropagation through sampling)
- **Decoder**: Linear → ReLU → Linear → Sigmoid (reconstructs TF-IDF vector)
- **Loss**: Reconstruction loss (MSE) + KL divergence (weighted by 0.001)

In [ ]:
import torch.nn as nn

class TextVAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.mu_layer = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        mu = self.mu_layer(encoded)
        logvar = self.logvar_layer(encoded)
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std
        reconstructed = self.decoder(z)
        return reconstructed, mu, logvar

model = TextVAE(input_dim=input_dim, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"VAE parameters: {total_params:,}")
print(f"Architecture: {input_dim} → {HIDDEN_DIM} → {LATENT_DIM} → {HIDDEN_DIM} → {input_dim}")

## 8. Train the VAE 🚀

Training uses only non-crisis samples. The loss combines:
- **Reconstruction loss (MSE)**: How well can the VAE reconstruct the input?
- **KL divergence**: Regularizes the latent space to be close to N(0,1)

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def iterate_batches(data, batch_size):
    for start in range(0, len(data), batch_size):
        yield data[start : start + batch_size]

model.train()
history = []

for epoch in range(EPOCHS):
    epoch_losses = []
    for batch in iterate_batches(x_train, BATCH_SIZE):
        tensor = torch.tensor(batch, dtype=torch.float32, device=device)
        reconstructed, mu, logvar = model(tensor)
        recon_loss = torch.mean((reconstructed - tensor) ** 2)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + 0.001 * kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    avg_loss = np.mean(epoch_losses)
    history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.6f}")

print("\n✅ Training complete!")

## 9. Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history) + 1), history, marker='o', linewidth=2, color='#2196F3')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('VAE Training Loss')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Compute Reconstruction Errors & Threshold

The threshold is set at the **95th percentile** of validation (non-crisis) reconstruction errors. This means:
- ~5% of normal text may be flagged (false positives)
- Crisis text should have errors **above** this threshold
- This is a **recall-oriented** setting — we'd rather flag too many than miss a real crisis

In [ ]:
from typing import List

def reconstruction_errors(model, data, device, batch_size=512):
    model.eval()
    errors: List[np.ndarray] = []
    with torch.no_grad():
        for batch in iterate_batches(data, batch_size):
            tensor = torch.tensor(batch, dtype=torch.float32, device=device)
            reconstructed, _, _ = model(tensor)
            per_row = torch.mean((reconstructed - tensor) ** 2, dim=1)
            errors.append(per_row.detach().cpu().numpy())
    return np.concatenate(errors, axis=0) if errors else np.array([], dtype=np.float32)

# Compute errors on all splits
val_errors = reconstruction_errors(model, x_val, device)
full_errors = reconstruction_errors(model, x_full, device)

# Determine threshold from validation (non-crisis) errors
threshold = float(np.percentile(val_errors, THRESHOLD_PERCENTILE))

print(f"Validation error — mean: {np.mean(val_errors):.6f}, std: {np.std(val_errors):.6f}")
print(f"Threshold (p{THRESHOLD_PERCENTILE}): {threshold:.6f}")

## 11. Evaluate: VAE vs Keyword Baseline

In [ ]:
# VAE predictions
predictions = (full_errors >= threshold).astype(int)
truth = df["is_crisis"].to_numpy(dtype=int)

# Keyword baseline
CRISIS_MARKERS = {
    "suicide", "kill myself", "end my life", "self harm",
    "can't go on", "cannot go on", "want to die", "no reason to live",
}
keyword_preds = df["text"].apply(
    lambda t: int(any(m in t.lower() for m in CRISIS_MARKERS))
).to_numpy()

def binary_metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn}

vae_metrics = binary_metrics(truth, predictions)
kw_metrics = binary_metrics(truth, keyword_preds)

print("=" * 50)
print("VAE Crisis Detector:")
print(f"  Precision: {vae_metrics['precision']:.4f}")
print(f"  Recall:    {vae_metrics['recall']:.4f}")
print(f"  F1:        {vae_metrics['f1']:.4f}")
print(f"  TP={vae_metrics['tp']}, FP={vae_metrics['fp']}, FN={vae_metrics['fn']}, TN={vae_metrics['tn']}")
print("\nKeyword Baseline:")
print(f"  Precision: {kw_metrics['precision']:.4f}")
print(f"  Recall:    {kw_metrics['recall']:.4f}")
print(f"  F1:        {kw_metrics['f1']:.4f}")
print(f"  TP={kw_metrics['tp']}, FP={kw_metrics['fp']}, FN={kw_metrics['fn']}, TN={kw_metrics['tn']}")
print("=" * 50)

## 12. Visualize Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution by class
crisis_mask = truth == 1
axes[0].hist(full_errors[~crisis_mask], bins=50, alpha=0.7, label='Non-crisis', color='#4CAF50', density=True)
axes[0].hist(full_errors[crisis_mask], bins=50, alpha=0.7, label='Crisis', color='#F44336', density=True)
axes[0].axvline(threshold, color='#FF9800', linestyle='--', linewidth=2, label=f'Threshold={threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Density')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()

# Comparison bar chart
methods = ['VAE', 'Keyword']
f1_vals = [vae_metrics['f1'], kw_metrics['f1']]
recall_vals = [vae_metrics['recall'], kw_metrics['recall']]
prec_vals = [vae_metrics['precision'], kw_metrics['precision']]

x = np.arange(len(methods))
width = 0.25
axes[1].bar(x - width, prec_vals, width, label='Precision', color='#2196F3')
axes[1].bar(x, recall_vals, width, label='Recall', color='#4CAF50')
axes[1].bar(x + width, f1_vals, width, label='F1', color='#FF9800')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_ylim(0, 1)
axes[1].set_title('VAE vs Keyword Baseline')
axes[1].legend()

fig.tight_layout()
fig.savefig('vae_evaluation_plots.png', dpi=150)
plt.show()
print("📊 Saved: vae_evaluation_plots.png")

## 13. Save Model Artifacts

Save the trained model, TF-IDF vocabulary, reconstruction errors, and threshold summary.

In [ ]:
import json
from datetime import datetime, timezone

output_dir = Path("vae_crisis_model")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Save model state dict
torch.save(model.state_dict(), output_dir / "vae_state_dict.pt")
print(f"✅ Saved: {output_dir / 'vae_state_dict.pt'}")

# 2. Save TF-IDF vocabulary (required by vae_detector.py)
vocab = {token: int(idx) for token, idx in vectorizer.vocabulary_.items()}
vocab_path = output_dir / "tfidf_vocabulary.json"
vocab_path.write_text(json.dumps(vocab, indent=2))
print(f"✅ Saved: {vocab_path} ({len(vocab):,} terms)")

# 3. Save reconstruction errors CSV
errors_df = pd.DataFrame({
    "text": df["text"].tolist(),
    "is_crisis": truth.tolist(),
    "reconstruction_error": full_errors.tolist(),
    "predicted_crisis": predictions.tolist(),
})
errors_path = output_dir / "reconstruction_errors.csv"
errors_df.to_csv(errors_path, index=False)
print(f"✅ Saved: {errors_path}")

# 4. Save threshold summary (required by vae_detector.py)
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_path": DATA_PATH,
    "counts": {
        "total": int(len(df)),
        "non_crisis_total": int(len(non_crisis)),
        "crisis_total": int(len(crisis)),
        "non_crisis_train": int(len(train_nc)),
        "non_crisis_validation": int(len(val_nc)),
    },
    "threshold": {
        "percentile": THRESHOLD_PERCENTILE,
        "value": threshold,
        "validation_error_mean": float(np.mean(val_errors)),
        "validation_error_std": float(np.std(val_errors)),
    },
    "metrics": vae_metrics,
    "keyword_baseline_metrics": kw_metrics,
    "artifacts": {
        "model_state": str((output_dir / "vae_state_dict.pt").as_posix()),
        "vectorizer_vocabulary": str(vocab_path.as_posix()),
        "reconstruction_errors": str(errors_path.as_posix()),
    },
    "training_config": {
        "max_features": MAX_FEATURES,
        "hidden_dim": HIDDEN_DIM,
        "latent_dim": LATENT_DIM,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "threshold_percentile": THRESHOLD_PERCENTILE,
    },
}

summary_path = output_dir / "vae_threshold_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"✅ Saved: {summary_path}")

print(f"\n📁 All artifacts saved to: {output_dir}/")

## 14. Quick Sanity Check

Verify the model detects crisis text correctly.

In [ ]:
def score_text(text, model, vectorizer, device):
    vec = vectorizer.transform([text]).toarray().astype(np.float32)
    tensor = torch.tensor(vec, dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        reconstructed, _, _ = model(tensor)
        error = torch.mean((reconstructed - tensor) ** 2, dim=1)
    return float(error.cpu().numpy().item())

test_cases = [
    ("I had a great day at work today", False),
    ("Feeling a bit tired but okay", False),
    ("I want to end my life", True),
    ("I cannot go on anymore, everything is hopeless", True),
    ("Thank you for being there for me", False),
    ("I feel like there is no reason to live", True),
]

print(f"Threshold: {threshold:.6f}")
print("=" * 70)
for text, expected_crisis in test_cases:
    error = score_text(text, model, vectorizer, device)
    detected = error >= threshold
    status = "✅" if detected == expected_crisis else "⚠️"
    print(f"{status} error={error:.6f} {'CRISIS' if detected else 'safe':>6s} | {text}")

## 15. Download Artifacts 📦

Download the model artifacts and place them in your project:

```
MindGuard/
├── models/
│   └── vae_crisis/                          ← Extract here
│       ├── vae_state_dict.pt
│       ├── tfidf_vocabulary.json
│       └── reconstruction_errors.csv
└── data/
    └── processed/
        └── vae_threshold_summary.json        ← Copy here
```

In [ ]:
import shutil

# Create a zip with everything
zip_path = shutil.make_archive("vae_crisis_model", "zip", ".", "vae_crisis_model")
print(f"✅ Created: {zip_path}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    # Also download the summary separately for easy placement
    files.download(str(summary_path))
    files.download("vae_evaluation_plots.png")
    print("📥 Downloads started!")
except ImportError:
    print(f"Not in Colab. Manually download: {zip_path}")

print("\n" + "=" * 60)
print("📋 SETUP INSTRUCTIONS")
print("=" * 60)
print("1. Extract vae_crisis_model.zip")
print("2. Copy these files into your MindGuard project:")
print("   • vae_state_dict.pt       → models/vae_crisis/vae_state_dict.pt")
print("   • tfidf_vocabulary.json   → models/vae_crisis/tfidf_vocabulary.json")
print("   • reconstruction_errors.csv → models/vae_crisis/reconstruction_errors.csv")
print("   • vae_threshold_summary.json → data/processed/vae_threshold_summary.json")
print("3. Restart your API server: uvicorn app.main:app --reload")
print("=" * 60)

## ✅ Done!

### Summary of trained model:
- **Architecture**: TextVAE (TF-IDF → 256 → 32 → 256 → TF-IDF)
- **Training data**: Non-crisis text only (anomaly detection approach)
- **Threshold**: 95th percentile of validation reconstruction errors
- **Safety policy**: Recall-oriented — prefers false positives over missed crises

### Next steps:
1. Place model files in your project (see instructions above)
2. Run `python scripts/evaluate_baselines.py` to generate evaluation artifacts
3. The crisis endpoint (`POST /predict/crisis`) will now use the trained VAE